# Module 10 — MLflow Experiment Tracking & MLOps
**PriceMind AI**

This notebook demonstrates the production-grade MLOps layer:
- Centralized MLflow Tracking
- Hyperparameter, Metric & Artifact Logging
- MLflow Model Registry & Model Promotion
- Model Loading & Version Verification
- SHAP Explainability & Prediction Compatibility


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ml.tracking import (
    setup_mlflow,
    get_tracking_uri,
    is_mlflow_enabled,
    EXPERIMENT_DEMAND_PREDICTION,
    get_or_create_experiment,
    set_active_experiment,
    ProductionModelLoader,
)

print(f"MLflow Version: {mlflow.__version__}")
print(f"Tracking URI: {get_tracking_uri()}")
print(f"Tracking Enabled: {is_mlflow_enabled()}")


2026/09/21 11:41:05 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\DELL\Desktop\My PROJECTS\PriceMind AI\backend\venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


MLflow Version: 3.16.1
Tracking URI: sqlite:///C:\Users\DELL\Desktop\My PROJECTS\PriceMind AI\mlruns.db
Tracking Enabled: True


In [2]:
setup_mlflow()
exp_id = set_active_experiment(EXPERIMENT_DEMAND_PREDICTION)
print(f"Active Experiment: {EXPERIMENT_DEMAND_PREDICTION} (ID: {exp_id})")


2026/09/21 11:41:10 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/21 11:41:10 INFO mlflow.store.db.utils: Updating database tables


Active Experiment: PriceMind-Demand-Prediction (ID: 1)


In [3]:
from ml.training.train_xgboost import train_and_track_xgboost

print("Training & Tracking XGBoost Regressor...")
xgb_results = train_and_track_xgboost(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.08,
    register_model=True,
    promote_to_production=True,
)

print(f"\nXGBoost Run ID: {xgb_results['run_id']}")
print(f"Registered Version: {xgb_results['registered_version']}")
print(f"Test RMSE: {xgb_results['test_metrics']['rmse']:.4f}")
print(f"Test R2: {xgb_results['test_metrics']['r2']:.4f}")


Training & Tracking XGBoost Regressor...


2026/09/21 11:41:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully registered model 'PriceMind-Demand-XGBoost'.
2026/09/21 11:41:38 WARNING mlflow.tracking._model_registry.fluent: Run with id b66a6765d9a74ecbb28055b53baa57a1 has no artifacts at artifact path 'model', registering model based on models:/m-57fa5d32d38d40a18fca7ae16c3e0381 instead



XGBoost Run ID: b66a6765d9a74ecbb28055b53baa57a1
Registered Version: 1
Test RMSE: 2.8752
Test R2: 0.9444


Created version '1' of model 'PriceMind-Demand-XGBoost'.


In [4]:
from ml.training.train_lightgbm import train_and_track_lightgbm

print("Training & Tracking LightGBM Regressor...")
lgb_results = train_and_track_lightgbm(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.08,
    register_model=True,
)

print(f"\nLightGBM Run ID: {lgb_results['run_id']}")
print(f"Registered Version: {lgb_results['registered_version']}")
print(f"Test RMSE: {lgb_results['test_metrics']['rmse']:.4f}")
print(f"Test R2: {lgb_results['test_metrics']['r2']:.4f}")


Training & Tracking LightGBM Regressor...


2026/09/21 11:41:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully registered model 'PriceMind-Demand-LightGBM'.
2026/09/21 11:42:03 WARNING mlflow.tracking._model_registry.fluent: Run with id 1d44f16b62ca4883a6b0f5348bbf14f0 has no artifacts at artifact path 'model', registering model based on models:/m-6d28c18e6aa24da58abba6526760ed2d instead



LightGBM Run ID: 1d44f16b62ca4883a6b0f5348bbf14f0
Registered Version: 1
Test RMSE: 2.8725
Test R2: 0.9445


Created version '1' of model 'PriceMind-Demand-LightGBM'.


In [5]:
from ml.training.evaluate_models import compare_mlflow_runs

comparison_df = compare_mlflow_runs()
print("=== Candidate Model Benchmark Comparison ===")
print(comparison_df[["run_id", "model_type", "test_rmse", "test_mae", "test_r2", "status"]].to_string(index=False))


=== Candidate Model Benchmark Comparison ===
  run_id               model_type  test_rmse  test_mae  test_r2   status
1d44f16b LightGBM (LGBMRegressor)     2.8725    1.9305   0.9445 FINISHED
b66a6765   XGBoost (XGBRegressor)     2.8752    1.9144   0.9444 FINISHED


In [6]:
loader = ProductionModelLoader()
model, version, source = loader.load()

print(f"Production Model Object: {type(model).__name__}")
print(f"Model Version: {version}")
print(f"Source: {source}")
print(f"Feature Count: {len(loader.feature_names)}")


[ModelLoader] MLflow registry load failed (Registered Model with name=PriceMind-Demand-Prediction not found). Falling back to local artifact.


Production Model Object: XGBRegressor
Model Version: v1
Source: local_artifact
Feature Count: 79


In [7]:
features_df = pd.read_parquet("../data/processed/features.parquet")
sample_row = features_df.tail(1)[loader.feature_names]

prediction = float(model.predict(sample_row)[0])
print(f"Sample Input Price: ${sample_row['price'].iloc[0]:.2f}")
print(f"Predicted Expected Demand: {prediction:.2f} units")


Sample Input Price: $380.53
Predicted Expected Demand: 15.61 units


In [8]:
from ml.explainability.explainer import ShapExplainer
from ml.explainability.local_explanations import LocalExplainer

explainer = ShapExplainer(model=model, feature_names=loader.feature_names).load()
local_exp = LocalExplainer(explainer)

explanation = local_exp.explain_prediction(sample_row, top_n=5)
print(f"Base Value E[f(X)]: {explanation.base_value:.2f}")
print(f"Reconstructed Prediction: {explanation.predicted_value:.2f}")
print(f"Additive Consistent: {explanation.additive_consistent}")
print(f"Top 3 Positive Drivers: {[(c.feature_name, round(c.shap_value, 3)) for c in explanation.top_positive_contributors[:3]]}")
print(f"Top 3 Negative Drivers: {[(c.feature_name, round(c.shap_value, 3)) for c in explanation.top_negative_contributors[:3]]}")


Base Value E[f(X)]: 18.89
Reconstructed Prediction: 15.62
Additive Consistent: True
Top 3 Positive Drivers: [(np.str_('demand_lag_14'), 0.462), (np.str_('day_of_week'), 0.429), (np.str_('week_sin'), 0.373)]
Top 3 Negative Drivers: [(np.str_('demand_lag_7'), -1.233), (np.str_('demand_rolling_mean_28'), -0.853), (np.str_('log_price'), -0.587)]


## Module 10 Summary
- MLflow tracking server and experiments initialized.
- XGBoost and LightGBM tracked with parameters, metrics, plots, and signatures.
- Production model registered and promoted with alias `production`.
- `ProductionModelLoader` provides seamless registry loading and local fallback.
- Predictions and SHAP explanations verify exact model version consistency.
